In [ ]:
import re
from pathlib import Path
from collections import Counter

def restore_batch_results(output_path):
    """从 batch_get 输出的 IR 文件还原 {(pass, loc): result}。"""
    output_path = Path(output_path)
    if not output_path.is_dir():
        raise NotADirectoryError(output_path)

    def parse_counts(value, expected_count, ir_path):
        numbers = [int(number) for number in re.findall(r"[+-]?\d+", value)]
        if len(numbers) != expected_count:
            raise ValueError(f"invalid result header in {ir_path}: {value}")
        return numbers

    all_results = {}
    for ir1_path in sorted(output_path.rglob("*-1.ll")):
        header = {}
        with ir1_path.open(encoding="utf-8") as f:
            for line in f:
                if line == "\n":
                    break
                if not line.startswith("; "):
                    raise ValueError(f"missing result header in {ir1_path}")
                name, separator, value = line[2:].rstrip("\n").partition(": ")
                if separator:
                    header[name] = value

        loc_path, loc_line = header["loc"].rsplit(":", 1)
        segment, segment_index = header["segment"].rsplit(",", 1)
        ir2_path = ir1_path.with_name(ir1_path.name.removesuffix("-1.ll") + "-2.ll")
        if not ir2_path.is_file():
            raise FileNotFoundError(ir2_path)

        result = {
            "id": header["id"],
            "func": header["func"],
            "pass": header["pass"],
            "segment": segment,
            "segment_index": int(segment_index),
            "loc": (None if loc_path == "None" else loc_path, int(loc_line)),
            "ir1_path": ir1_path.resolve(),
            "ir2_path": ir2_path.resolve(),
        }

        result["L_orig"], result["L_new"] = parse_counts(header["L"], 2, ir1_path)
        for name in ("BB", "Inst", "NBB", "NI"):
            orig, added, removed, modified = parse_counts(header[name], 4, ir1_path)
            result[f"{name}_orig"] = orig
            result[f"{name}_add"] = added
            result[f"{name}_del"] = abs(removed)
            result[f"{name}_mod"] = modified
        result["NEBB_add"], nebb_removed = parse_counts(header["NEBB"], 2, ir1_path)
        result["NEBB_del"] = abs(nebb_removed)
        if header["destroyed"] not in ("True", "False"):
            raise ValueError(f"invalid destroyed value in {ir1_path}")
        result["destroyed"] = header["destroyed"] == "True"

        key = (result["pass"], result["loc"])
        old_result = all_results.get(key)
        if old_result is not None :
            old_score = (old_result["NI_add"] + old_result["NI_del"])
            new_score = (result["NI_add"] + result["NI_del"])
            if new_score > old_score:
                print(
                    f"WARNING: duplicate key {key} {result['id']}-{result['func']}, "
                    f"replacing result {old_result['id']}-{old_result['func']} "
                    f"because (NI_add + NI_del) increased from "
                    f"{old_score:.6g} to {new_score:.6g}"
                )
                all_results[key] = result
            else:
                print(
                    f"WARNING: duplicate key {key} {result['id']}-{result['func']}, "
                    f"keeping result {old_result['id']}-{old_result['func']} "
                    f"because (NI_add + NI_del) decreased from "
                    f"{new_score:.6g} to {old_score:.6g}"
                )
        else:  
            all_results[key] = result

    return all_results

if __name__ == "__main__":
    output_path = "test/260715/ll"
    r_restore = restore_batch_results(output_path)
    # print(r_restore==rO3)
    pass_counter = Counter(pass_name for (pass_name, loc) in r_restore)
    for pass_name in sorted(pass_counter):
        print(f"{pass_name}: {pass_counter[pass_name]}")

In [ ]:
import subprocess
llvm_path = "/home/dxic/llvmlearn/llvm-project/llvm/build/bin/"

def get_obj_from_ir(ir, binary_path):
    cmd_list = [
        llvm_path+ "clang",
        "-x",
        "ir",
        "-O0", 
        "-c",
        # "-fno-pie",
        "-o", 
        binary_path,
        "-",
    ]

    subprocess.run(cmd_list, check=True, input=ir, encoding="utf-8")

if __name__ == "__main__":
    with open("test/twofuncO3.ll", "r") as f:
        ir = f.read()
    binary_path = "test/twofuncO3.o"
    get_obj_from_ir(ir, binary_path)

In [ ]:
def rename_global_value(ir, old_name, new_name):
    cmd = [
        "/home/dxic/LoopOptDiff2/LoopOptDataset/build/ir-rename", 
        old_name, 
        new_name
    ]
    result = subprocess.run(cmd, input=ir, check=True, stdout=subprocess.PIPE, text=True)
    return result.stdout

if __name__ == "__main__":
    # print(ir_g)
    old_name = "z"
    new_name = "zz"
    print(rename_global_value(ir, old_name, new_name))
    

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def compile_ir_func_to_obj(func_name, new_func_name, ir_path):
    ir_path = Path(ir_path).resolve()
    if not ir_path.is_file():
        raise FileNotFoundError(ir_path)

    ir = ir_path.read_text(encoding="utf-8")
    renamed_ir = rename_global_value(ir, func_name, new_func_name)
    obj_path = ir_path.with_suffix(".o")
    get_obj_from_ir(renamed_ir, obj_path)
    return obj_path

if __name__ == "__main__":
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {}
        for results in r_selected_result.values():
            for result in results:
                new_func_name = f"{result['id']}_{result['func']}"
                for obj_path_key, ir_path_key in (("obj1_path", "ir1_path"), ("obj2_path", "ir2_path")):
                    future = executor.submit(
                        compile_ir_func_to_obj,
                        result["func"],
                        new_func_name,
                        result[ir_path_key],
                    )
                    futures[future] = (result, obj_path_key)

        for future, (result, obj_path_key) in futures.items():
            result[obj_path_key] = future.result()

    for passname in passname_list:
        print(passname, len(r_selected_result[passname]))
        print(r_selected_result[passname][0]["obj1_path"])
        print(r_selected_result[passname][0]["obj2_path"])


In [ ]:
# 按 NI 绝对大小和变化比例二维分层随机选择样本，只复制 IR，不编译或链接二进制

from collections import defaultdict
from pathlib import Path
import random
import shutil

batch_result_path = "./test/260831/ll"
output_path = "./test/260831/ratio2"
n = 1000
size_strata = 5
ratio_strata = 10
seed = 42
passname_list = None  # None 表示选择 batch_result_path 中的全部 pass
r_selected_result = {}

def select_by_size_ratio_distribution(results, n, size_strata=5, ratio_strata=10, seed=42):
    """按 NI_orig 和 r=(NI_add+NI_del)/NI_orig 的二维连续分位层抽样。"""
    result_values = list(results.values()) if isinstance(results, dict) else list(results)
    if n < 0:
        raise ValueError("n must be non-negative")
    if size_strata <= 0:
        raise ValueError("size_strata must be positive")
    if ratio_strata <= 0:
        raise ValueError("ratio_strata must be positive")
    invalid = [
        result
        for result in result_values
        if result.get("NI_orig") is None or result["NI_orig"] <= 0
    ]
    if invalid:
        examples = ', '.join(
            f"{result.get('id', '<unknown>')}-{result.get('func', '<unknown>')}"
            for result in invalid[:5]
        )
        raise ValueError(
            f"{len(invalid)} samples have NI_orig <= 0 and cannot compute ratio"
            f" (examples: {examples})"
        )
    sample_size = min(n, len(result_values))
    if sample_size == 0:
        return []

    ordered = [
        (
            result["NI_orig"],
            (result["NI_add"] + result["NI_del"]) / result["NI_orig"],
            index,
            dict(result),
        )
        for index, result in enumerate(result_values)
    ]
    ordered.sort(key=lambda item: (item[0], item[2]))
    size_count = min(size_strata, len(ordered))
    size_base, size_extra = divmod(len(ordered), size_count)
    size_layers = []
    start = 0
    for layer_index in range(size_count):
        layer_size = size_base + (layer_index < size_extra)
        size_layers.append(ordered[start:start + layer_size])
        start += layer_size

    strata = []
    for size_layer in size_layers:
        size_layer.sort(key=lambda item: (item[1], item[2]))
        ratio_count = min(ratio_strata, len(size_layer))
        ratio_base, ratio_extra = divmod(len(size_layer), ratio_count)
        start = 0
        for ratio_index in range(ratio_count):
            layer_size = ratio_base + (ratio_index < ratio_extra)
            strata.append(size_layer[start:start + layer_size])
            start += layer_size

    raw_quotas = [sample_size * len(stratum) / len(ordered) for stratum in strata]
    quotas = [min(len(stratum), int(quota)) for stratum, quota in zip(strata, raw_quotas)]
    remaining = sample_size - sum(quotas)
    remainder_order = sorted(
        range(len(strata)),
        key=lambda index: (raw_quotas[index] - int(raw_quotas[index]), -index),
        reverse=True,
    )
    for stratum_index in remainder_order:
        if remaining == 0:
            break
        if quotas[stratum_index] < len(strata[stratum_index]):
            quotas[stratum_index] += 1
            remaining -= 1
    if remaining:
        raise RuntimeError("could not allocate the exact sample size")

    rng = random.Random(seed)
    selected = []
    for stratum, quota in zip(strata, quotas):
        selected.extend(rng.sample(stratum, quota))
    selected.sort(key=lambda item: item[2])
    return [result for _, _, _, result in selected]



if __name__ == "__main__":
    restored = restore_batch_results(batch_result_path)
    results_by_pass = defaultdict(list)
    for result in restored.values():
        results_by_pass[result["pass"]].append(result)

    if passname_list is None:
        passname_list = sorted(results_by_pass)
    else:
        passname_list = list(passname_list)
    missing_passes = sorted(set(passname_list) - set(results_by_pass))
    if missing_passes:
        raise ValueError(f"passes not found in batch results: {missing_passes}")

    # 先完成全部 pass 的验证和选择，避免部分复制后才发现输入不合法。
    selected_by_pass = {}
    for passname in passname_list:
        results = results_by_pass[passname]
        try:
            selected_by_pass[passname] = select_by_size_ratio_distribution(
                results, n, size_strata=size_strata, ratio_strata=ratio_strata, seed=seed
            )
        except ValueError as exc:
            raise ValueError(f"{passname}: {exc}") from exc

    root = Path(output_path).resolve()
    root.mkdir(parents=True, exist_ok=True)
    for passname, selected in selected_by_pass.items():
        pass_dir = root / passname
        if pass_dir.exists():
            if not pass_dir.is_dir():
                raise NotADirectoryError(pass_dir)
            if any(pass_dir.iterdir()):
                raise FileExistsError(f"ratio selection output directory is not empty: {pass_dir}")
        else:
            pass_dir.mkdir(parents=True)

        target_names = set()
        for result in selected:
            for path_key, suffix in (("ir1_path", "-1.ll"), ("ir2_path", "-2.ll")):
                source = Path(result[path_key]).resolve()
                if not source.is_file():
                    raise FileNotFoundError(source)
                if not source.name.endswith(suffix):
                    raise ValueError(f"unexpected IR filename: {source}")
                if source.name in target_names:
                    raise FileExistsError(f"IR target filename conflict: {pass_dir / source.name}")
                target_names.add(source.name)

        for result in selected:
            for path_key in ("ir1_path", "ir2_path"):
                source = Path(result[path_key]).resolve()
                target = (pass_dir / source.name).resolve()
                shutil.copy2(source, target)
                result[path_key] = target

        r_selected_result[passname] = selected
        print(f"{passname}: selected {len(selected)}/{len(results_by_pass[passname])}")


In [ ]:
# 从选择的1000函数样本构建Binary similarity method的输入

import random

relink_root = Path("./test/260831/ratio2")
# passname_list = None # None 表示选择 batch_result_path 中的全部 pass
passname_list = ["IndVarSimplifyPass", "LICMPass", "LoopFullUnrollPass", 
                     "LoopIdiomRecognizePass", "LoopLoadEliminationPass", "LoopRotatePass", 
                     "LoopStrengthReducePass", "LoopUnrollPass", "LoopVectorizePass", 
                     "SimpleLoopUnswitchPass"]
relink_shuffle_seed = 42
relink_pipeline_outputs = {}
num_workers = 16

if __name__ == "__main__":
    if passname_list is None:
        passname_list = sorted(results_by_pass)
    else:
        passname_list = list(passname_list)
    missing_passes = sorted(set(passname_list) - set(results_by_pass))
    if missing_passes:
        raise ValueError(f"passes not found in batch results: {missing_passes}")

    for passname in passname_list:
        # 恢复 IR，补编译缺失的单对象。
        pass_dir = (relink_root / passname).resolve()
        if not pass_dir.is_dir():
            raise NotADirectoryError(pass_dir)
        restored = restore_batch_results(pass_dir)
        compile_tasks = []
        for result in restored.values():
            for obj_key, ir_key in (("obj1_path", "ir1_path"), ("obj2_path", "ir2_path")):
                obj_path = Path(result[ir_key]).with_suffix(".o")
                if not obj_path.is_file():
                    compile_tasks.append((
                        result["func"],
                        f"{result['id']}_{result['func']}",
                        result[ir_key],
                    ))

        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [
                executor.submit(compile_ir_func_to_obj, *task)
                for task in compile_tasks
            ]
            for future in futures:
                future.result()
        selected = restore_selected_ir(pass_dir)
        if not selected:
            raise ValueError(f"selected results are empty: {pass_dir}")

        print(f"{passname}: {len(selected)} samples")
